In [ ]:
import sys
import os
import importlib
from pathlib import Path
from datetime import datetime
importlib.invalidate_caches()

target_path = '/content/drive/MyDrive/Thesis/models'
if target_path not in sys.path:
    sys.path.insert(0, target_path)

import time
import argparse
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, DataCollatorWithPadding,
)
from sklearn.metrics import f1_score, accuracy_score

from utils import DATA_DIR, MODELS_DIR, EKMAN_LABELS, set_all_seeds
from evaluate import compute_metrics, save_metrics, plot_confusion_matrix
from train_phase3 import (
    GoEmotionsDataset, hf_compute_metrics, load_data,
    MODEL_NAME, MAX_LENGTH, BATCH_SIZE, N_EPOCHS, LR, WEIGHT_DECAY, WARMUP,
    NUM_LABELS,
)

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

from transformers import Trainer


def compute_class_weights(
    labels,
    num_labels=NUM_LABELS,
) -> torch.Tensor:
    labels = np.asarray(
        labels,
        dtype=np.int64,
    )

    if labels.ndim != 1:
        raise ValueError(
            "Class labels must be a one-dimensional array."
        )

    if len(labels) == 0:
        raise ValueError(
            "Cannot compute class weights from an empty training-label array. "
        )

    if labels.min() < 0:
        raise ValueError(
            "Class labels cannot contain negative values."
        )

    if labels.max() >= num_labels:
        raise ValueError(
            f"Found class ID {labels.max()}, but NUM_LABELS "
            f"is {num_labels}. Valid IDs are "
            f"0 through {num_labels - 1}."
        )

    class_counts = np.bincount(
        labels,
        minlength=num_labels,
    )

    missing_classes = np.where(
        class_counts == 0
    )[0]

    if len(missing_classes) > 0:
        raise ValueError(
            "The following classes are absent from the "
            f"training set: {missing_classes.tolist()}. "
            "Class weights cannot be calculated safely."
        )

    number_of_examples = len(labels)
    number_of_classes = num_labels

    class_weights = (
        number_of_examples
        / (
            number_of_classes
            * class_counts
        )
    )

    print("\nTraining class counts:")
    for class_id, count in enumerate(
        class_counts
    ):
        class_name = EKMAN_LABELS[class_id]

        print(
            f"{class_id}: {class_name:<10} "
            f"count={count:<6} "
            f"weight={class_weights[class_id]:.6f}"
        )

    return torch.tensor(
        class_weights,
        dtype=torch.float32,
    )


class WeightedCETrainer(Trainer):

    def __init__(
        self,
        *args,
        class_weights=None,
        **kwargs,
    ):
        if class_weights is None:
            raise ValueError(
                "class_weights must be provided to "
                "WeightedCETrainer."
            )

        super().__init__(
            *args,
            **kwargs,
        )

        if not isinstance(
            class_weights,
            torch.Tensor,
        ):
            class_weights = torch.tensor(
                class_weights,
                dtype=torch.float32,
            )

        if class_weights.ndim != 1:
            raise ValueError(
                "class_weights must be one-dimensional."
            )

        if len(class_weights) != NUM_LABELS:
            raise ValueError(
                f"Expected {NUM_LABELS} class weights, "
                f"but received {len(class_weights)}."
            )

        if not torch.all(
            torch.isfinite(class_weights)
        ):
            raise ValueError(
                "Class weights contain NaN or infinity."
            )

        if torch.any(class_weights <= 0):
            raise ValueError(
                "Every class weight must be positive."
            )

        self.class_weights = (
            class_weights
            .detach()
            .clone()
            .to(dtype=torch.float32)
        )

        self.model_accepts_loss_kwargs = False

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
        **kwargs,
    ):
        model_inputs = dict(inputs)

        labels = model_inputs.pop(
            "labels"
        ).long()

        outputs = model(
            **model_inputs
        )

        logits = outputs.logits

        if logits.ndim != 2:
            raise ValueError(
                "Expected logits with shape "
                "[batch_size, NUM_LABELS], but received "
                f"{tuple(logits.shape)}."
            )

        if logits.shape[-1] != NUM_LABELS:
            raise ValueError(
                f"Expected {NUM_LABELS} output logits, "
                f"but received {logits.shape[-1]}."
            )

        weights = self.class_weights.to(
            device=logits.device,
        )

        loss = F.cross_entropy(
            input=logits,
            target=labels,
            weight=weights,
            reduction="mean",
        )

        return (
            (loss, outputs)
            if return_outputs
            else loss
        )

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import Trainer


class FocalLoss(nn.Module):

    def __init__(
        self,
        gamma: float = 2.0,
        alpha: float = 1.0,
        reduction: str = "mean",
    ):
        super().__init__()

        if gamma < 0:
            raise ValueError(
                "gamma must be greater than or equal to zero."
            )

        if alpha <= 0:
            raise ValueError(
                "alpha must be greater than zero."
            )

        if reduction not in {
            "none",
            "mean",
            "sum",
        }:
            raise ValueError(
                "reduction must be 'none', 'mean', or 'sum'."
            )

        self.gamma = float(gamma)
        self.alpha = float(alpha)
        self.reduction = reduction

    def forward(
        self,
        logits: torch.Tensor,
        targets: torch.Tensor,
    ) -> torch.Tensor:

        if logits.ndim != 2:
            raise ValueError(
                "Expected logits with shape "
                "[batch_size, number_of_classes], "
                f"but received {tuple(logits.shape)}."
            )

        if targets.ndim != 1:
            raise ValueError(
                "Expected targets with shape [batch_size], "
                f"but received {tuple(targets.shape)}."
            )

        if logits.shape[0] != targets.shape[0]:
            raise ValueError(
                "The number of logits and target labels "
                "does not match."
            )

        number_of_classes = logits.shape[-1]

        if torch.any(targets < 0):
            raise ValueError(
                "Targets contain negative class IDs."
            )

        if torch.any(targets >= number_of_classes):
            raise ValueError(
                "A target class ID is larger than the number "
                "of model output classes."
            )

        targets = targets.long()

        log_probabilities = F.log_softmax(
            logits,
            dim=-1,
        )

        log_pt = log_probabilities.gather(
            dim=1,
            index=targets.unsqueeze(1),
        ).squeeze(1)

        pt = log_pt.exp()

        pt = pt.clamp(
            min=0.0,
            max=1.0,
        )

        focal_factor = (
            1.0 - pt
        ).pow(self.gamma)

        per_example_loss = (
            -self.alpha
            * focal_factor
            * log_pt
        )

        if self.reduction == "none":
            return per_example_loss

        if self.reduction == "sum":
            return per_example_loss.sum()

        return per_example_loss.mean()


class FocalTrainer(Trainer):

    def __init__(
        self,
        *args,
        gamma: float = 2.0,
        alpha: float = 1.0,
        **kwargs,
    ):
        super().__init__(
            *args,
            **kwargs,
        )

        self.loss_fn = FocalLoss(
            gamma=gamma,
            alpha=alpha,
            reduction="mean",
        )

        self.model_accepts_loss_kwargs = False

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
        **kwargs,
    ):

        model_inputs = dict(inputs)

        labels = model_inputs.pop(
            "labels"
        ).long()

        outputs = model(
            **model_inputs
        )

        logits = outputs.logits

        if logits.shape[-1] != NUM_LABELS:
            raise ValueError(
                f"Expected {NUM_LABELS} model outputs, "
                f"but received {logits.shape[-1]}."
            )

        loss = self.loss_fn(
            logits=logits,
            targets=labels,
        )

        return (
            (loss, outputs)
            if return_outputs
            else loss
        )

In [ ]:
import gc
import json
from transformers import MarianMTModel, MarianTokenizer


def _translate_deterministically(
    texts,
    tokenizer,
    model,
    device,
    batch_size=8,
    max_length=128,
):
    translations = []

    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            batch = texts[start:start + batch_size]

            encoded = tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=max_length,
            ).to(device)

            generated = model.generate(
                **encoded,
                max_length=max_length,
                num_beams=4,
                do_sample=False,
            )

            translations.extend(
                tokenizer.batch_decode(
                    generated,
                    skip_special_tokens=True,
                )
            )

    return translations


def _generate_diverse_reverse_translations(
    pivot_texts,
    original_texts,
    tokenizer,
    model,
    device,
    aug_factor=3,
    candidates_per_source=12,
    batch_size=8,
    max_length=128,
    seed=42,
    maximum_attempts=8,
):
    if candidates_per_source < aug_factor:
        raise ValueError(
            "candidates_per_source must be greater than or equal "
            "to aug_factor."
        )

    if maximum_attempts < 1:
        raise ValueError(
            "maximum_attempts must be at least 1."
        )

    selected_paraphrases = []

    temperature_schedule = [
        0.80,
        0.90,
        1.00,
        1.10,
        1.20,
        1.30,
        1.40,
        1.50,
    ]

    top_p_schedule = [
        0.90,
        0.93,
        0.95,
        0.96,
        0.97,
        0.98,
        0.98,
        0.99,
    ]

    with torch.no_grad():
        for start in range(
            0,
            len(pivot_texts),
            batch_size,
        ):
            pivot_batch = pivot_texts[
                start:start + batch_size
            ]

            original_batch = original_texts[
                start:start + batch_size
            ]

            candidate_pools = [
                [] for _ in pivot_batch
            ]

            for attempt in range(maximum_attempts):
                if all(
                    len(pool) >= aug_factor
                    for pool in candidate_pools
                ):
                    break

                current_seed = (
                    seed
                    + start * 100
                    + attempt
                )

                torch.manual_seed(current_seed)

                if torch.cuda.is_available():
                    torch.cuda.manual_seed_all(
                        current_seed
                    )

                temperature = temperature_schedule[
                    min(
                        attempt,
                        len(temperature_schedule) - 1,
                    )
                ]

                top_p = top_p_schedule[
                    min(
                        attempt,
                        len(top_p_schedule) - 1,
                    )
                ]

                encoded = tokenizer(
                    pivot_batch,
                    return_tensors="pt",
                    padding=True,
                    truncation=True,
                    max_length=max_length,
                ).to(device)

                generated = model.generate(
                    **encoded,
                    max_length=max_length,
                    do_sample=True,
                    num_beams=1,
                    top_p=top_p,
                    top_k=50,
                    temperature=temperature,
                    repetition_penalty=1.05,
                    num_return_sequences=(
                        candidates_per_source
                    ),
                )

                decoded = tokenizer.batch_decode(
                    generated,
                    skip_special_tokens=True,
                )

                for local_index, original_text in enumerate(
                    original_batch
                ):
                    if (
                        len(candidate_pools[local_index])
                        >= aug_factor
                    ):
                        continue

                    begin = (
                        local_index
                        * candidates_per_source
                    )

                    end = (
                        begin
                        + candidates_per_source
                    )

                    normalized_original = (
                        " ".join(
                            original_text
                            .strip()
                            .casefold()
                            .split()
                        )
                    )

                    existing_normalized_candidates = {
                        " ".join(
                            existing
                            .strip()
                            .casefold()
                            .split()
                        )
                        for existing
                        in candidate_pools[local_index]
                    }

                    for candidate in decoded[begin:end]:
                        candidate = " ".join(
                            candidate.strip().split()
                        )

                        if not candidate:
                            continue

                        normalized_candidate = (
                            candidate.casefold()
                        )

                        if (
                            normalized_candidate
                            == normalized_original
                        ):
                            continue

                        if (
                            normalized_candidate
                            in existing_normalized_candidates
                        ):
                            continue

                        candidate_pools[
                            local_index
                        ].append(candidate)

                        existing_normalized_candidates.add(
                            normalized_candidate
                        )

                        if (
                            len(
                                candidate_pools[
                                    local_index
                                ]
                            )
                            >= aug_factor
                        ):
                            break

                incomplete_count = sum(
                    len(pool) < aug_factor
                    for pool in candidate_pools
                )

                print(
                    f"Batch beginning at source {start}: "
                    f"attempt {attempt + 1}/"
                    f"{maximum_attempts}; "
                    f"{incomplete_count} sources still "
                    f"need more paraphrases."
                )

            for local_index, pool in enumerate(
                candidate_pools
            ):
                source_index = start + local_index

                if len(pool) < aug_factor:
                    original_text = original_batch[
                        local_index
                    ]

                    raise RuntimeError(
                        "\nUnable to generate enough distinct "
                        "back-translations.\n"
                        f"Source row: {source_index}\n"
                        f"Original text: {original_text!r}\n"
                        f"Distinct paraphrases generated: "
                        f"{len(pool)}\n"
                        f"Required paraphrases: "
                        f"{aug_factor}\n"
                        f"Candidates requested per attempt: "
                        f"{candidates_per_source}\n"
                        f"Attempts: {maximum_attempts}\n\n"
                        "The code stopped rather than inserting "
                        "duplicate or unchanged sentences."
                    )

                selected_paraphrases.append(
                    pool[:aug_factor]
                )

    if len(selected_paraphrases) != len(original_texts):
        raise RuntimeError(
            "The number of generated paraphrase groups does not "
            "match the number of original source texts."
        )

    return selected_paraphrases


def augment_minority_classes(
    train_df,
    top_k=3,
    aug_factor=3,
    candidates_per_source=12,
    maximum_attempts=8,
    pivot="de",
    batch_size=8,
    seed=42,
):

    import gc

    from transformers import (
        MarianMTModel,
        MarianTokenizer,
    )

    required_columns = {
        "text_clean",
        "ekman_id",
    }

    if not required_columns.issubset(
        train_df.columns
    ):
        raise ValueError(
            f"The training dataframe must contain "
            f"{required_columns}."
        )

    class_counts = (
        train_df["ekman_id"]
        .value_counts()
        .sort_values(ascending=True)
    )

    number_of_classes = len(class_counts)

    if not 1 <= top_k < number_of_classes:
        raise ValueError(
            f"top_k must be between 1 and "
            f"{number_of_classes - 1}. "
            f"The dataset contains "
            f"{number_of_classes} classes, "
            f"but top_k={top_k} was requested."
        )

    selected_class_ids = (
        class_counts
        .index[:top_k]
        .astype(int)
        .tolist()
    )

    print("\nOriginal training-class counts:")
    print(
        class_counts
        .sort_index()
        .to_string()
    )

    print(
        f"\nSelected {top_k} rarest classes: "
        f"{selected_class_ids}"
    )

    selected_df = (
        train_df[
            train_df["ekman_id"].isin(
                selected_class_ids
            )
        ]
        .reset_index(drop=True)
        .copy()
    )

    source_texts = (
        selected_df["text_clean"]
        .astype(str)
        .tolist()
    )

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    forward_name = (
        f"Helsinki-NLP/opus-mt-en-{pivot}"
    )

    reverse_name = (
        f"Helsinki-NLP/opus-mt-{pivot}-en"
    )

    print(f"\nLoading {forward_name}...")

    forward_tokenizer = (
        MarianTokenizer.from_pretrained(
            forward_name
        )
    )

    forward_model = (
        MarianMTModel
        .from_pretrained(forward_name)
        .to(device)
        .eval()
    )

    print(
        "Translating English into the pivot "
        "language..."
    )

    pivot_texts = _translate_deterministically(
        texts=source_texts,
        tokenizer=forward_tokenizer,
        model=forward_model,
        device=device,
        batch_size=batch_size,
    )

    del forward_model
    del forward_tokenizer

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(f"\nLoading {reverse_name}...")

    reverse_tokenizer = (
        MarianTokenizer.from_pretrained(
            reverse_name
        )
    )

    reverse_model = (
        MarianMTModel
        .from_pretrained(reverse_name)
        .to(device)
        .eval()
    )

    print(
        "Generating diverse English "
        "back-translations..."
    )

    paraphrase_groups = (
        _generate_diverse_reverse_translations(
            pivot_texts=pivot_texts,
            original_texts=source_texts,
            tokenizer=reverse_tokenizer,
            model=reverse_model,
            device=device,
            aug_factor=aug_factor,
            candidates_per_source=(
                candidates_per_source
            ),
            batch_size=batch_size,
            seed=seed,
            maximum_attempts=maximum_attempts,
        )
    )

    extra_rows = []

    for row_index, paraphrases in enumerate(
        paraphrase_groups
    ):
        class_id = int(
            selected_df.iloc[row_index][
                "ekman_id"
            ]
        )

        source_text = source_texts[
            row_index
        ]

        for paraphrase_number, paraphrase in enumerate(
            paraphrases,
            start=1,
        ):
            extra_rows.append({
                "text_clean": paraphrase,
                "ekman_id": class_id,
                "augmentation":
                    "back_translation",
                "source_text": source_text,
                "paraphrase_number":
                    paraphrase_number,
            })

    extra_df = pd.DataFrame(extra_rows)

    expected_examples = (
        len(selected_df) * aug_factor
    )

    if len(extra_df) != expected_examples:
        raise RuntimeError(
            f"Expected {expected_examples} synthetic "
            f"examples, but generated "
            f"{len(extra_df)}."
        )


    augmented_df = train_df.copy()

    augmented_df["augmentation"] = "original"
    augmented_df["source_text"] = (
        augmented_df["text_clean"]
    )
    augmented_df["paraphrase_number"] = 0

    augmented_df = pd.concat(
        [
            augmented_df,
            extra_df,
        ],
        ignore_index=True,
    )

    print(
        f"\nGenerated {len(extra_df):,} "
        f"distinct synthetic examples."
    )

    print("\nAugmented training-class counts:")

    augmented_counts = (
        augmented_df["ekman_id"]
        .value_counts()
        .sort_index()
    )

    print(augmented_counts.to_string())

    augmentation_report = {
        "method": "English-German-English "
                  "back-translation",
        "pivot_language": pivot,
        "diversity_method":
            "top-p sampling during reverse translation",
        "top_k": top_k,
        "aug_factor": aug_factor,
        "candidates_per_source":
            candidates_per_source,
        "maximum_attempts":
            maximum_attempts,
        "selected_class_ids":
            selected_class_ids,
        "original_training_size":
            int(len(train_df)),
        "selected_source_examples":
            int(len(selected_df)),
        "expected_synthetic_examples":
            int(expected_examples),
        "generated_synthetic_examples":
            int(len(extra_df)),
        "augmented_training_size":
            int(len(augmented_df)),
        "original_class_counts": {
            str(int(class_id)): int(count)
            for class_id, count
            in class_counts.sort_index().items()
        },
        "augmented_class_counts": {
            str(int(class_id)): int(count)
            for class_id, count
            in augmented_counts.items()
        },
    }

    del reverse_model
    del reverse_tokenizer

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return (
        augmented_df,
        extra_df,
        augmentation_report,
    )

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import confusion_matrix
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)


def make_json_safe(value):
    if isinstance(value, dict):
        return {
            str(key): make_json_safe(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [make_json_safe(item) for item in value]

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, np.generic):
        return value.item()

    return value


def save_confusion_matrix_to_folder(
    y_true,
    y_pred,
    labels,
    output_path,
    title,
):
    matrix = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(len(labels))),
    )

    figure, axis = plt.subplots(figsize=(9, 8))

    image = axis.imshow(
        matrix,
        interpolation="nearest",
        cmap="Blues",
    )

    figure.colorbar(image, ax=axis)

    axis.set(
        xticks=np.arange(len(labels)),
        yticks=np.arange(len(labels)),
        xticklabels=labels,
        yticklabels=labels,
        xlabel="Predicted label",
        ylabel="True label",
        title=title,
    )

    plt.setp(
        axis.get_xticklabels(),
        rotation=45,
        ha="right",
        rotation_mode="anchor",
    )

    threshold = matrix.max() / 2 if matrix.size else 0

    for row_index in range(matrix.shape[0]):
        for column_index in range(matrix.shape[1]):
            axis.text(
                column_index,
                row_index,
                format(matrix[row_index, column_index], "d"),
                ha="center",
                va="center",
                color=(
                    "white"
                    if matrix[row_index, column_index] > threshold
                    else "black"
                ),
            )

    figure.tight_layout()
    figure.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(figure)

    return matrix


def run(
    treatment: str,
    seed: int = 42,
    gamma: float = 2.0,
    alpha: float = 1.0,
    top_k: int = 3,
    aug_factor: int = 3,
    candidates_per_source: int = 12,
    maximum_attempts: int = 8,
    output_root=None,
) -> dict:
    set_all_seeds(seed)

    if treatment not in {"cw", "focal", "bt"}:
        raise ValueError(
            "treatment must be 'cw', 'focal', or 'bt'."
        )

    if treatment == "cw":
        basic_run_name = f"phase4_cw_seed{seed}"

    elif treatment == "focal":
        basic_run_name = (
            f"phase4_focal_g{gamma:g}_seed{seed}"
        )

    else:
        basic_run_name = (
            f"phase4_bt_k{top_k}_x{aug_factor}_seed{seed}"
        )

    if output_root is None:
        raise ValueError(
            "You must provide output_root. For example:\n"
            "output_root='/content/drive/MyDrive/Thesis/"
            "corrected_phase4_runs'"
        )

    output_root = Path(output_root)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_name = f"{basic_run_name}_{timestamp}"
    out_dir = output_root / run_name

    if out_dir.exists():
        raise FileExistsError(
            f"The following output folder already exists:\n"
            f"{out_dir}\n"
            "No files were overwritten."
        )

    out_dir.mkdir(
        parents=True,
        exist_ok=False,
    )

    print("=" * 70)
    print("PHASE 4 EXPERIMENT")
    print("=" * 70)
    print(f"Treatment: {treatment}")
    print(f"Seed: {seed}")
    print(f"Run name: {run_name}")
    print(f"All new results will be saved in:\n{out_dir}")
    print("=" * 70)

    train_df, val_df, test_df = load_data()

    original_training_size = len(train_df)

    print("\nOriginal dataset sizes:")
    print(f"Training:   {len(train_df):,}")
    print(f"Validation: {len(val_df):,}")
    print(f"Test:       {len(test_df):,}")

    original_class_counts = (
        train_df["ekman_id"]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    synthetic_df = None
    augmentation_report = None

    if treatment == "bt":
        print("\nStarting P4c back-translation...")
        translation_start_time = time.time()

        (
            train_df,
            synthetic_df,
            augmentation_report,
        ) = augment_minority_classes(
            train_df=train_df,
            top_k=top_k,
            aug_factor=aug_factor,
            candidates_per_source=candidates_per_source,
            maximum_attempts=maximum_attempts,
            pivot="de",
            batch_size=8,
            seed=seed,
        )
        translation_time = (
            time.time() - translation_start_time
        )

        print(
            f"\nBack-translation completed in "
            f"{translation_time:.1f} seconds "
            f"({translation_time / 60:.2f} minutes)."
        )

        synthetic_df.to_csv(
            out_dir / "generated_paraphrases.csv",
            index=False,
        )

        with open(
            out_dir / "augmentation_report.json",
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(
                make_json_safe(augmentation_report),
                file,
                indent=2,
                ensure_ascii=False,
            )

    else:
        print(
            f"\nNo back-translation applied to "
            f"treatment '{treatment}'."
        )

    final_training_size = len(train_df)

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        id2label={
            index: label
            for index, label in enumerate(EKMAN_LABELS)
        },
        label2id={
            label: index
            for index, label in enumerate(EKMAN_LABELS)
        },
    )

    train_ds = GoEmotionsDataset(
        train_df["text_clean"],
        train_df["ekman_id"],
        tokenizer,
    )

    val_ds = GoEmotionsDataset(
        val_df["text_clean"],
        val_df["ekman_id"],
        tokenizer,
    )

    test_ds = GoEmotionsDataset(
        test_df["text_clean"],
        test_df["ekman_id"],
        tokenizer,
    )

    training_args = TrainingArguments(
        output_dir=str(out_dir / "checkpoints"),
        learning_rate=LR,
        weight_decay=WEIGHT_DECAY,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=N_EPOCHS,
        warmup_steps=WARMUP,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=2,
        logging_steps=100,
        seed=seed,
        data_seed=seed,
        report_to="none",
    )

    common_arguments = dict(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(
            tokenizer=tokenizer
        ),
        compute_metrics=hf_compute_metrics,
    )

    if treatment == "cw":
        weights = compute_class_weights(
            train_df["ekman_id"].values
        )

        print("\nClass weights:")
        for class_index, weight in enumerate(weights.tolist()):
            print(
                f"{EKMAN_LABELS[class_index]}: "
                f"{weight:.6f}"
            )

        trainer = WeightedCETrainer(
            class_weights=weights,
            **common_arguments,
        )

    elif treatment == "focal":
        trainer = FocalTrainer(
            gamma=gamma,
            alpha=alpha,
            **common_arguments,
        )

    else:
        trainer = Trainer(
            **common_arguments
        )

    print("\nStarting training...")

    training_start_time = time.time()

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    train_result = trainer.train()

    training_time = time.time() - training_start_time

    print(
        f"\nTraining completed in {training_time:.1f} seconds "
        f"({training_time / 60:.2f} minutes)."
    )

    peak_gpu_memory_gb = None

    if torch.cuda.is_available():
        peak_gpu_memory_gb = (
            torch.cuda.max_memory_allocated() / 1e9
        )

        print(
            f"Peak training GPU memory: "
            f"{peak_gpu_memory_gb:.2f} GB"
        )

    best_model_dir = out_dir / "best_model"

    trainer.save_model(
        str(best_model_dir)
    )

    tokenizer.save_pretrained(
        str(best_model_dir)
    )

    def evaluate_and_save(
        dataset,
        dataframe,
        split_name,
    ):
        print(f"\nEvaluating {split_name} set...")

        prediction_output = trainer.predict(
            dataset,
            metric_key_prefix=split_name,
        )

        logits = prediction_output.predictions

        if isinstance(logits, tuple):
            logits = logits[0]

        y_true = prediction_output.label_ids.astype(int)
        y_pred = np.argmax(logits, axis=-1).astype(int)

        stable_logits = (
            logits - np.max(logits, axis=1, keepdims=True)
        )

        exponentials = np.exp(stable_logits)

        probabilities = (
            exponentials
            / np.sum(
                exponentials,
                axis=1,
                keepdims=True,
            )
        )

        split_metrics = compute_metrics(
            y_true,
            y_pred,
            EKMAN_LABELS,
        )

        predictions_dataframe = pd.DataFrame({
            "id": dataframe["id"].tolist(),
            "text_clean": dataframe[
                "text_clean"
            ].astype(str).tolist(),
            "true_id": y_true,
            "true_label": [
                EKMAN_LABELS[value]
                for value in y_true
            ],
            "predicted_id": y_pred,
            "predicted_label": [
                EKMAN_LABELS[value]
                for value in y_pred
            ],
            "correct": y_true == y_pred,
        })

        for class_index, class_name in enumerate(
            EKMAN_LABELS
        ):
            safe_class_name = (
                str(class_name)
                .lower()
                .replace(" ", "_")
            )

            predictions_dataframe[
                f"probability_{safe_class_name}"
            ] = probabilities[:, class_index]

        predictions_dataframe.to_csv(
            out_dir / f"{split_name}_predictions.csv",
            index=False,
        )

        confusion_matrix_values = (
            save_confusion_matrix_to_folder(
                y_true=y_true,
                y_pred=y_pred,
                labels=EKMAN_LABELS,
                output_path=(
                    out_dir
                    / f"{split_name}_confusion_matrix.png"
                ),
                title=(
                    f"{split_name.capitalize()} confusion matrix"
                ),
            )
        )

        np.savetxt(
            out_dir
            / f"{split_name}_confusion_matrix.csv",
            confusion_matrix_values,
            delimiter=",",
            fmt="%d",
        )

        with open(
            out_dir / f"{split_name}_metrics.json",
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(
                make_json_safe(split_metrics),
                file,
                indent=2,
                ensure_ascii=False,
            )

        return split_metrics

    validation_metrics = evaluate_and_save(
        dataset=val_ds,
        dataframe=val_df,
        split_name="validation",
    )

    test_metrics = evaluate_and_save(
        dataset=test_ds,
        dataframe=test_df,
        split_name="test",
    )

    final_class_counts = (
        train_df["ekman_id"]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    complete_results = {
        "run_name": run_name,
        "treatment": treatment,
        "seed": seed,
        "model_name": MODEL_NAME,
        "top_k": top_k if treatment == "bt" else None,
        "augmentation_factor": (
            aug_factor if treatment == "bt" else None
        ),
        "candidates_per_source": (
            candidates_per_source
            if treatment == "bt"
            else None
        ),
        "gamma": (
            gamma if treatment == "focal" else None
        ),
        "alpha": (
            alpha if treatment == "focal" else None
        ),
        "original_training_size":
            original_training_size,
        "final_training_size":
            final_training_size,
        "original_training_class_counts":
            original_class_counts,
        "final_training_class_counts":
            final_class_counts,
        "best_checkpoint":
            trainer.state.best_model_checkpoint,
        "best_validation_macro_f1_during_training":
            trainer.state.best_metric,
        "training_time_seconds":
            training_time,
        "peak_gpu_memory_gb":
            peak_gpu_memory_gb,
        "augmentation":
            augmentation_report,
        "validation":
            validation_metrics,
        "test":
            test_metrics,
    }

    complete_results_path = (
        out_dir / "complete_results.json"
    )

    with open(
        complete_results_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            make_json_safe(complete_results),
            file,
            indent=2,
            ensure_ascii=False,
        )

    with open(
        out_dir / "training_log_history.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            make_json_safe(trainer.state.log_history),
            file,
            indent=2,
            ensure_ascii=False,
        )

    print("\n" + "=" * 70)
    print("FINAL RESULTS")
    print("=" * 70)

    print("\nValidation metrics:")
    print(
        json.dumps(
            make_json_safe(validation_metrics),
            indent=2,
        )
    )

    print("\nTest metrics:")
    print(
        json.dumps(
            make_json_safe(test_metrics),
            indent=2,
        )
    )

    print(f"\nBest checkpoint:")
    print(trainer.state.best_model_checkpoint)

    print("\nAll results were saved in:")
    print(out_dir)

    print("\nNo original dataset files were modified.")
    print("=" * 70)

    return complete_results

In [ ]:
P4C_OUTPUT_LOCATION = (
    "/content/drive/MyDrive/Thesis/models_new"
)

p4c_results = run(
    treatment="bt",
    seed=42,
    top_k=3,
    aug_factor=3,
    candidates_per_source=12,
    maximum_attempts=8,
    output_root=P4C_OUTPUT_LOCATION,
)

PHASE 4 EXPERIMENT
Treatment: bt
Seed: 42
Run name: phase4_bt_k3_x3_seed42_20260619_100401
All new results will be saved in:
/content/drive/MyDrive/Thesis/models_new/phase4_bt_k3_x3_seed42_20260619_100401

Original dataset sizes:
Training:   36,302
Validation: 4,547
Test:       4,590

Starting P4c back-translation...

Original training-class counts:
ekman_id
0     3878
1      497
2      515
3    12920
4     2121
5     3553
6    12818

Selected 3 rarest classes: [1, 2, 4]

Loading Helsinki-NLP/opus-mt-en-de...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Translating English into the pivot language...

Loading Helsinki-NLP/opus-mt-de-en...


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

...


Training completed in 525.7 seconds (8.76 minutes).
Peak training GPU memory: 2.18 GB


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Evaluating validation set...



Evaluating test set...



FINAL RESULTS

Validation metrics:
{
  "accuracy": 0.686826478997141,
  "macro_f1": 0.5980211106599319,
  "weighted_f1": 0.6859895758851762,
  "per_class_f1": [
    0.45614035087719296,
    0.45348837209302323,
    0.6356589147286822,
    0.8065635847270433,
    0.5730337078651685,
    0.572463768115942,
    0.6887990762124712
  ],
  "report": {
    "anger": {
      "precision": 0.5814696485623003,
      "recall": 0.3752577319587629,
      "f1-score": 0.45614035087719296,
      "support": 485.0
    },
    "disgust": {
      "precision": 0.35135135135135137,
      "recall": 0.639344262295082,
      "f1-score": 0.45348837209302323,
      "support": 61.0
    },
    "fear": {
      "precision": 0.6507936507936508,
      "recall": 0.6212121212121212,
      "f1-score": 0.6356589147286822,
      "support": 66.0
    },
    "joy": {
      "precision": 0.8514323784143905,
      "recall": 0.7661870503597122,
      "f1-score": 0.8065635847270433,
      "support": 1668.0
    },
    "sadness": {
  

In [ ]:
P4C_OUTPUT_LOCATION = (
    "/content/drive/MyDrive/Thesis/models_new"
)

p4c_seed_0_results = run(
    treatment="bt",
    seed=0,
    top_k=3,
    aug_factor=3,
    candidates_per_source=12,
    maximum_attempts=8,
    output_root=P4C_OUTPUT_LOCATION,
)

p4c_seed_1_results = run(
    treatment="bt",
    seed=1,
    top_k=3,
    aug_factor=3,
    candidates_per_source=12,
    maximum_attempts=8,
    output_root=P4C_OUTPUT_LOCATION,
)


PHASE 4 EXPERIMENT
Treatment: bt
Seed: 0
Run name: phase4_bt_k3_x3_seed0_20260619_114814
All new results will be saved in:
/content/drive/MyDrive/Thesis/models_new/phase4_bt_k3_x3_seed0_20260619_114814

Original dataset sizes:
Training:   36,302
Validation: 4,547
Test:       4,590

Starting P4c back-translation...

Original training-class counts:
ekman_id
0     3878
1      497
2      515
3    12920
4     2121
5     3553
6    12818

Selected 3 rarest classes: [1, 2, 4]

Loading Helsinki-NLP/opus-mt-en-de...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/768k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/797k [00:00<?, ?B/s]

...


Training completed in 488.0 seconds (8.13 minutes).
Peak training GPU memory: 2.18 GB


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Evaluating validation set...



Evaluating test set...



FINAL RESULTS

Validation metrics:
{
  "accuracy": 0.6659335825819221,
  "macro_f1": 0.5872452781985764,
  "weighted_f1": 0.6733496569535082,
  "per_class_f1": [
    0.47297297297297297,
    0.430939226519337,
    0.6762589928057554,
    0.8040907638223075,
    0.4930747922437673,
    0.5717592592592593,
    0.6616209397666352
  ],
  "report": {
    "anger": {
      "precision": 0.5210918114143921,
      "recall": 0.4329896907216495,
      "f1-score": 0.47297297297297297,
      "support": 485.0
    },
    "disgust": {
      "precision": 0.325,
      "recall": 0.639344262295082,
      "f1-score": 0.430939226519337,
      "support": 61.0
    },
    "fear": {
      "precision": 0.6438356164383562,
      "recall": 0.7121212121212122,
      "f1-score": 0.6762589928057554,
      "support": 66.0
    },
    "joy": {
      "precision": 0.8610540725530459,
      "recall": 0.7541966426858513,
      "f1-score": 0.8040907638223075,
      "support": 1668.0
    },
    "sadness": {
      "precision":

In [ ]:
P4A_OUTPUT_LOCATION = (
  "/content/drive/MyDrive/Thesis/models_new"
)

p4a_seed_42_results = run(
    treatment="cw",
    seed=42,
    output_root=P4A_OUTPUT_LOCATION,
)

p4a_seed_0_results = run(
    treatment="cw",
    seed=0,
    output_root=P4A_OUTPUT_LOCATION,
)

p4a_seed_1_results = run(
    treatment="cw",
    seed=1,
    output_root=P4A_OUTPUT_LOCATION,
)

PHASE 4 EXPERIMENT
Treatment: cw
Seed: 42
Run name: phase4_cw_seed42_20260619_131809
All new results will be saved in:
/content/drive/MyDrive/Thesis/models_new/phase4_cw_seed42_20260619_131809

Original dataset sizes:
Training:   36,302
Validation: 4,547
Test:       4,590

No back-translation applied to treatment 'cw'.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Training class counts:
0: anger      count=3878   weight=1.337287
1: disgust    count=497    weight=10.434608
2: fear       count=515    weight=10.069903
3: joy        count=12920  weight=0.401393
4: sadness    count=2121   weight=2.445073
5: surprise   count=3553   weight=1.459612
6: neutral    count=12818  weight=0.404587

Class weights:
anger: 1.337287
disgust: 10.434608
fear: 10.069903
joy: 0.401393
sadness: 2.445073
surprise: 1.459612
neutral: 0.404587

Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,1.009883,0.990144,0.619969,0.551378,0.635107
2,0.874000,0.959762,0.664614,0.601089,0.672570
3,0.666932,1.005059,0.660216,0.595925,0.666353


...


Training completed in 399.9 seconds (6.66 minutes).
Peak training GPU memory: 3.19 GB


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Evaluating validation set...



Evaluating test set...



FINAL RESULTS

Validation metrics:
{
  "accuracy": 0.6632944798768419,
  "macro_f1": 0.6034143653534988,
  "weighted_f1": 0.6694198230236954,
  "per_class_f1": [
    0.5283993115318416,
    0.45454545454545453,
    0.6459627329192547,
    0.8072251635004671,
    0.5916955017301038,
    0.5837231057062675,
    0.6123492875411034
  ],
  "report": {
    "anger": {
      "precision": 0.4534711964549483,
      "recall": 0.6329896907216495,
      "f1-score": 0.5283993115318416,
      "support": 485.0
    },
    "disgust": {
      "precision": 0.34782608695652173,
      "recall": 0.6557377049180327,
      "f1-score": 0.45454545454545453,
      "support": 61.0
    },
    "fear": {
      "precision": 0.5473684210526316,
      "recall": 0.7878787878787878,
      "f1-score": 0.6459627329192547,
      "support": 66.0
    },
    "joy": {
      "precision": 0.8399222294232016,
      "recall": 0.7769784172661871,
      "f1-score": 0.8072251635004671,
      "support": 1668.0
    },
    "sadness": {
 

In [ ]:
P4B_GAMMA2_OUTPUT_LOCATION = (
        "/content/drive/MyDrive/Thesis/models_new"
)

p4b_g2_seed_42_results = run(
    treatment="focal",
    gamma=2.0,
    seed=42,
    output_root=P4B_GAMMA2_OUTPUT_LOCATION,
)

p4b_g2_seed_0_results = run(
    treatment="focal",
    gamma=2.0,
    seed=0,
    output_root=P4B_GAMMA2_OUTPUT_LOCATION,
)

p4b_g2_seed_1_results = run(
    treatment="focal",
    gamma=2.0,
    seed=1,
    output_root=P4B_GAMMA2_OUTPUT_LOCATION,
)

PHASE 4 EXPERIMENT
Treatment: focal
Seed: 42
Run name: phase4_focal_g2_seed42_20260619_134728
All new results will be saved in:
/content/drive/MyDrive/Thesis/models_new/phase4_focal_g2_seed42_20260619_134728

Original dataset sizes:
Training:   36,302
Validation: 4,547
Test:       4,590

No back-translation applied to treatment 'focal'.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.464178,0.427954,0.691885,0.620910,0.688310
2,0.387423,0.416927,0.700462,0.642988,0.699787
3,0.305916,0.426091,0.699802,0.640032,0.699380


...


Training completed in 399.5 seconds (6.66 minutes).
Peak training GPU memory: 3.19 GB


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Evaluating validation set...



Evaluating test set...



FINAL RESULTS

Validation metrics:
{
  "accuracy": 0.7004618429733891,
  "macro_f1": 0.6341205453661475,
  "weighted_f1": 0.6999801631165703,
  "per_class_f1": [
    0.5501022494887525,
    0.5196850393700787,
    0.6511627906976745,
    0.8186650915534555,
    0.6328600405679513,
    0.5981941309255079,
    0.6681744749596122
  ],
  "report": {
    "anger": {
      "precision": 0.5456389452332657,
      "recall": 0.554639175257732,
      "f1-score": 0.5501022494887525,
      "support": 485.0
    },
    "disgust": {
      "precision": 0.5,
      "recall": 0.5409836065573771,
      "f1-score": 0.5196850393700787,
      "support": 61.0
    },
    "fear": {
      "precision": 0.6666666666666666,
      "recall": 0.6363636363636364,
      "f1-score": 0.6511627906976745,
      "support": 66.0
    },
    "joy": {
      "precision": 0.8067520372526193,
      "recall": 0.8309352517985612,
      "f1-score": 0.8186650915534555,
      "support": 1668.0
    },
    "sadness": {
      "precision": 0

In [ ]:
P4B_GAMMA1_OUTPUT_LOCATION = (
        "/content/drive/MyDrive/Thesis/models_new"
)

p4b_g1_seed_42_results = run(
    treatment="focal",
    gamma=1.0,
    seed=42,
    output_root=P4B_GAMMA1_OUTPUT_LOCATION,
)

p4b_g1_seed_0_results = run(
    treatment="focal",
    gamma=1.0,
    seed=0,
    output_root=P4B_GAMMA1_OUTPUT_LOCATION,
)

p4b_g1_seed_1_results = run(
    treatment="focal",
    gamma=1.0,
    seed=1,
    output_root=P4B_GAMMA1_OUTPUT_LOCATION,
)

PHASE 4 EXPERIMENT
Treatment: focal
Seed: 42
Run name: phase4_focal_g1_seed42_20260619_140902
All new results will be saved in:
/content/drive/MyDrive/Thesis/models_new/phase4_focal_g1_seed42_20260619_140902

Original dataset sizes:
Training:   36,302
Validation: 4,547
Test:       4,590

No back-translation applied to treatment 'focal'.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.616179,0.573875,0.692545,0.617229,0.689611
2,0.523421,0.563470,0.700022,0.642032,0.699363
3,0.432654,0.575028,0.703541,0.645926,0.703072


...


Training completed in 409.5 seconds (6.82 minutes).
Peak training GPU memory: 3.19 GB


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Evaluating validation set...



Evaluating test set...



FINAL RESULTS

Validation metrics:
{
  "accuracy": 0.7000219925225424,
  "macro_f1": 0.6345630010451043,
  "weighted_f1": 0.6997620638590203,
  "per_class_f1": [
    0.541033434650456,
    0.532258064516129,
    0.6412213740458015,
    0.8168097070139094,
    0.6421267893660532,
    0.5975197294250282,
    0.6709719082983533
  ],
  "report": {
    "anger": {
      "precision": 0.5318725099601593,
      "recall": 0.5505154639175258,
      "f1-score": 0.541033434650456,
      "support": 485.0
    },
    "disgust": {
      "precision": 0.5238095238095238,
      "recall": 0.5409836065573771,
      "f1-score": 0.532258064516129,
      "support": 61.0
    },
    "fear": {
      "precision": 0.6461538461538462,
      "recall": 0.6363636363636364,
      "f1-score": 0.6412213740458015,
      "support": 66.0
    },
    "joy": {
      "precision": 0.8065458796025716,
      "recall": 0.8273381294964028,
      "f1-score": 0.8168097070139094,
      "support": 1668.0
    },
    "sadness": {
      "p